In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [9]:
pip install gdown

Note: you may need to restart the kernel to use updated packages.


In [3]:
!gdown --id 1Jgvm9CoDENCduRHsK0THD6Vr-Nf7aurp

/opt/conda/lib/python3.10/site-packages/gdown/cli.py:126: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (uriginal): https://drive.google.com/uc?id=1Jgvm9CoDENCduRHsK0THD6Vr-Nf7aurp
From (redirected): https://drive.google.com/uc?id=1Jgvm9CoDENCduRHsK0THD6Vr-Nf7aurp&confirm=t&uuid=a9ce7c74-7a80-46f0-b80e-2914574cdfe3
To: /kaggle/working/gee_features_10pct.csv
100%|████████████████████████████████████████| 876M/876M [00:13<00:00, 64.2MB/s]


In [10]:
!gdown --id 1BnihrfCbHh_oWm14E3iWimcvoJld7e3R

/opt/conda/lib/python3.10/site-packages/gdown/cli.py:126: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1BnihrfCbHh_oWm14E3iWimcvoJld7e3R
To: /kaggle/working/training_label.csv
100%|██████████████████████████████████████| 7.77M/7.77M [00:00<00:00, 31.1MB/s]


In [11]:
!gdown --id 1UCeIlCf9_KPD3Gu7TLODYob0N1C06tIi

/opt/conda/lib/python3.10/site-packages/gdown/cli.py:126: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1UCeIlCf9_KPD3Gu7TLODYob0N1C06tIi
To: /kaggle/working/sample submission.csv
100%|████████████████████████████████████████| 510k/510k [00:00<00:00, 94.0MB/s]


# Dimensionality Reduction of GEE Features File

In [12]:
import pandas as pd

chunk_size = 10000  # Set the desired chunk size

# Missing Value Ratio
threshold_missing = 0.8  # Set threshold for missing value ratio

# Low Variance Filter
threshold_variance = 0.1  # Set threshold for variance

# High Correlation Filter
threshold_correlation = 0.9  # Set threshold for correlation

# List to store filtered chunks
filtered_chunks = []

# Iterate over the dataset in chunks
for chunk in pd.read_csv('/kaggle/working/gee_features_10pct.csv', chunksize=chunk_size, low_memory = False):
    # Missing Value Ratio
    missing_ratio = chunk.isnull().sum() / len(chunk)
    missing_cols = missing_ratio[missing_ratio > threshold_missing].index
    chunk = chunk.drop(missing_cols, axis=1)

    # Low Variance Filter
    variances = chunk.var(numeric_only=True)
    low_variance_cols = variances[variances < threshold_variance].index
    chunk = chunk.drop(low_variance_cols, axis=1)

    # High Correlation Filter
    corr_matrix = chunk.corr(numeric_only=True).abs()
    upper_tri = corr_matrix.where(~np.tril(np.ones(corr_matrix.shape)).astype(bool))

    high_correlation_cols = [col for col in upper_tri.columns if any(upper_tri[col] > threshold_correlation)]
    chunk = chunk.drop(high_correlation_cols, axis=1)

    # Append the filtered chunk to the list
    filtered_chunks.append(chunk)

# Combine filtered chunks back together
combined_data = pd.concat(filtered_chunks, axis=0)

# Get the consistent feature set
common_features = set.intersection(*[set(chunk.columns) for chunk in filtered_chunks])
combined_data = combined_data[list(common_features)]

combined_data.head()

,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,LAI_QA_flag_median@JAXA/GCOM-C/L3/LAND/LAI/V2&timestamped,...,Retrieved_Temperature_Profile_Mean_Mean_950_median@MODIS/061/MOD08_M3&timestamped,sur_refl_b01_max_min@MODIS/006/MOD13A1&timestamped,Mean_mean@Oxford/MAP/LST_Night_5km_Monthly&timestamped,BRDF_Albedo_Band_Mandatory_Quality_Band1_mean@MODIS/006/MCD43A1&timestamped,Retrieved_Temperature_Profile_Std_Deviation_Mean_10_min_max@MODIS/061/MOD08_M3&timestamped,Cloud_Water_Path_1621_PCL_Ice_Mean_Mean_max_max@MODIS/061/MOD08_M3&timestamped,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped
0,50.0,16513.363,-2.997677,934.0,64.916664,-1.0,138,-2.99828,-1.000000,0,...,14637.0,2100,21.275425,0.186448,-14951,6,1533.6666,1554.0,-2.965393,0.303478
1,208.0,16514.078,-2.997677,811.5,64.416664,-1.0,140,-2.99828,-1.000000,0,...,14467.5,1502,-17.246767,0.527480,-14946,6,1130.8334,1329.0,-2.965393,-0.266114
2,156.5,16514.094,-1.437762,821.0,79.583336,-1.0,136,-2.99828,14.082428,0,...,14517.0,1801,19.879644,0.413432,-14950,6,1233.5000,888.0,-0.835440,-0.171740
3,215.5,16514.088,-2.997677,1464.0,103.250000,-1.0,139,-2.99828,-1.000000,0,...,14720.5,1513,20.646397,0.314884,-14937,6,1526.5000,1186.0,-2.965393,0.104836
4,245.0,16514.053,-2.997677,1369.0,112.250000,-1.0,139,-2.99828,-1.000000,0,...,14568.0,1923,18.503235,0.474879,-14924,6,1589.7277,2677.0,-2.965393,-0.255735


In [13]:
combined_data.shape
#(120984, 901)

(12098, 2395)

In [14]:
# combined_data.to_csv('gee_features_Dimenreduced.csv' , index = False)

In [15]:
# from IPython.display import FileLink
# FileLink(r'gee_features_Dimenreduced.csv')

/kaggle/working/gee_features_Dimenreduced.csv

## Splitting the reduced GEE Features file into Traning Data and Sample Data

In [ ]:
#gee_reduced = pd.read_csv('/kaggle/input/dimenreduced/gee_features_Dimenreduced.csv', low_memory = False)
#gee_reduced.head()

#here, gee_reduced is same as combined_data above

In [2]:
# combined_data = pd.read_csv('/kaggle/input/data-10pct/gee_features_Dimenreduced.csv')

In [3]:
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder object
label_encoder = LabelEncoder()

# Iterate over each column in the DataFrame
for column in combined_data.columns:
    if column != 'DHSID' and combined_data[column].dtype == 'object':  # Check if the column is non-numeric and not 'DHSID'
        combined_data[column] = label_encoder.fit_transform(combined_data[column].astype(str))

# Print the updated DataFrame
combined_data.head()

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,LAI_QA_flag_median@JAXA/GCOM-C/L3/LAND/LAI/V2&timestamped,...,Retrieved_Temperature_Profile_Mean_Mean_950_median@MODIS/061/MOD08_M3&timestamped,sur_refl_b01_max_min@MODIS/006/MOD13A1&timestamped,Mean_mean@Oxford/MAP/LST_Night_5km_Monthly&timestamped,BRDF_Albedo_Band_Mandatory_Quality_Band1_mean@MODIS/006/MCD43A1&timestamped,Retrieved_Temperature_Profile_Std_Deviation_Mean_10_min_max@MODIS/061/MOD08_M3&timestamped,Cloud_Water_Path_1621_PCL_Ice_Mean_Mean_max_max@MODIS/061/MOD08_M3&timestamped,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped
0,50.0,16513.363,-2.997677,934.0,64.916664,-1.0,138,-2.99828,-1.000000,0,...,14637.0,2100,21.275425,0.186448,-14951,6,1533.6666,1554.0,-2.965393,0.303478
1,208.0,16514.078,-2.997677,811.5,64.416664,-1.0,140,-2.99828,-1.000000,0,...,14467.5,1502,-17.246767,0.527480,-14946,6,1130.8334,1329.0,-2.965393,-0.266114
2,156.5,16514.094,-1.437762,821.0,79.583336,-1.0,136,-2.99828,14.082428,0,...,14517.0,1801,19.879644,0.413432,-14950,6,1233.5000,888.0,-0.835440,-0.171740
3,215.5,16514.088,-2.997677,1464.0,103.250000,-1.0,139,-2.99828,-1.000000,0,...,14720.5,1513,20.646397,0.314884,-14937,6,1526.5000,1186.0,-2.965393,0.104836
4,245.0,16514.053,-2.997677,1369.0,112.250000,-1.0,139,-2.99828,-1.000000,0,...,14568.0,1923,18.503235,0.474879,-14924,6,1589.7277,2677.0,-2.965393,-0.255735


In [4]:
dhsid = combined_data['DHSID']
dhsid

0        IA201400110884
1        IA201400051523
2        IA201400150534
3        ML200600000390
4        NG200300000174
              ...      
12093    PE200400001013
12094    EG200800000765
12095    PE200000000348
12096    PE200000000614
12097    DR200700001622
Name: DHSID, Length: 12098, dtype: object

In [5]:
from sklearn.impute import SimpleImputer

# Create a copy of the DataFrame without the 'DHSID' column
gee_reduced_no_dhsid = combined_data.drop('DHSID', axis=1)

# Create a SimpleImputer object
imputer = SimpleImputer(strategy='mean')

# Impute missing values without including the 'DHSID' column
df_imputed_no_dhsid = pd.DataFrame(imputer.fit_transform(gee_reduced_no_dhsid), columns=gee_reduced_no_dhsid.columns)

# Combine the imputed data with the 'DHSID' column
df_imputed = pd.concat([dhsid, df_imputed_no_dhsid], axis=1)

# Print the imputed DataFrame
df_imputed.head()

,DHSID,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,...,Retrieved_Temperature_Profile_Mean_Mean_950_median@MODIS/061/MOD08_M3&timestamped,sur_refl_b01_max_min@MODIS/006/MOD13A1&timestamped,Mean_mean@Oxford/MAP/LST_Night_5km_Monthly&timestamped,BRDF_Albedo_Band_Mandatory_Quality_Band1_mean@MODIS/006/MCD43A1&timestamped,Retrieved_Temperature_Profile_Std_Deviation_Mean_10_min_max@MODIS/061/MOD08_M3&timestamped,Cloud_Water_Path_1621_PCL_Ice_Mean_Mean_max_max@MODIS/061/MOD08_M3&timestamped,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped
0,IA201400110884,50.0,16513.363,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,...,14637.0,2100.0,21.275425,0.186448,-14951.0,6.0,1533.6666,1554.0,-2.965393,0.303478
1,IA201400051523,208.0,16514.078,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,...,14467.5,1502.0,-17.246767,0.527480,-14946.0,6.0,1130.8334,1329.0,-2.965393,-0.266114
2,IA201400150534,156.5,16514.094,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,...,14517.0,1801.0,19.879644,0.413432,-14950.0,6.0,1233.5000,888.0,-0.835440,-0.171740
3,ML200600000390,215.5,16514.088,-2.997677,1464.0,103.250000,-1.0,139.0,-2.99828,-1.000000,...,14720.5,1513.0,20.646397,0.314884,-14937.0,6.0,1526.5000,1186.0,-2.965393,0.104836
4,NG200300000174,245.0,16514.053,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,...,14568.0,1923.0,18.503235,0.474879,-14924.0,6.0,1589.7277,2677.0,-2.965393,-0.255735


In [6]:
#sample_sub = pd.read_csv('/kaggle/input/maternal-and-child-health-monitoring-in-lmics/sample submission.csv', low_memory = False)
#training_label = pd.read_csv('/kaggle/input/maternal-and-child-health-monitoring-in-lmics/training_label.csv', low_memory = False)

In [12]:
sample_sub = pd.read_csv('/kaggle/working/sample submission.csv', low_memory = False)
training_label = pd.read_csv('/kaggle/working/training_label.csv', low_memory = False)

In [13]:
gee_reduced_training = pd.merge(df_imputed , training_label.drop(columns=['LONGNUM', 'URBAN_RURA', 'LATNUM']) ,  on ='DHSID', how='inner')
gee_reduced_sample = pd.merge(df_imputed, sample_sub, on = 'DHSID', how = 'inner')
gee_reduced_training.head()

,DHSID,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,...,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,DHSYEAR_y,DHSCLUST_y,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,IA201400110884,50.0,16513.363,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,...,-2.965393,0.303478,2014,110884.0,21.67,20.44,42.86,NaN,NaN,NaN
1,IA201400051523,208.0,16514.078,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,...,-2.965393,-0.266114,2014,51523.0,19.62,20.22,66.67,NaN,NaN,NaN
2,IA201400150534,156.5,16514.094,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,...,-0.835440,-0.171740,2014,150534.0,20.07,20.83,83.33,NaN,NaN,NaN
3,NG200300000174,245.0,16514.053,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,...,-2.965393,-0.255735,2003,174.0,21.64,20.07,66.67,16.33,43.75,NaN
4,PH200800000205,173.0,16514.428,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,...,-2.965393,-0.360048,2008,205.0,NaN,NaN,33.33,2.78,87.50,NaN


In [14]:
gee_reduced_training.shape
#(101140, 909)

(10058, 2403)

In [15]:
gee_reduced_training.head()

,DHSID,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,...,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,DHSYEAR_y,DHSCLUST_y,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,IA201400110884,50.0,16513.363,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,...,-2.965393,0.303478,2014,110884.0,21.67,20.44,42.86,NaN,NaN,NaN
1,IA201400051523,208.0,16514.078,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,...,-2.965393,-0.266114,2014,51523.0,19.62,20.22,66.67,NaN,NaN,NaN
2,IA201400150534,156.5,16514.094,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,...,-0.835440,-0.171740,2014,150534.0,20.07,20.83,83.33,NaN,NaN,NaN
3,NG200300000174,245.0,16514.053,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,...,-2.965393,-0.255735,2003,174.0,21.64,20.07,66.67,16.33,43.75,NaN
4,PH200800000205,173.0,16514.428,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,...,-2.965393,-0.360048,2008,205.0,NaN,NaN,33.33,2.78,87.50,NaN


In [16]:
gee_reduced_training['DHSID'].value_counts()

DHSID
DR200700000259    2
DR200700000345    2
ZM201300000707    2
ZA201700000007    2
ZA201700000011    2
                 ..
IA201400340509    1
ZW201500000025    1
IA201400210537    1
PE200000000385    1
DR200700001622    1
Name: count, Length: 10029, dtype: int64

In [17]:
gee_reduced_sample.shape
#(15346, 907)

(1592, 2401)

In [18]:
gee_reduced_sample.drop_duplicates(subset = ['DHSID'] , inplace = True , keep = 'first')
gee_reduced_sample.shape

(1587, 2401)

In [19]:
missing_dhsids = list(set(sample_sub.DHSID.values) - set(gee_reduced_sample.DHSID.values))
missing_dhsids

'''['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']'''

"['EG201403480201',\n 'EG201407490204',\n 'DHS20180000402',\n 'EG201406870404',\n 'EG201403980103',\n 'EG201404452003',\n 'EG201407480104',\n 'EG201405500103',\n 'DHS20180000554',\n 'DHS20180000452',\n 'EG201403230104',\n 'EG201404100304',\n 'EG201404350102',\n 'EG201404460201',\n 'EG201407240101',\n 'EG201406992212',\n 'EG201406490204',\n 'DHS20180000579',\n 'EG201407110104',\n 'EG201404950402',\n 'EG201406740602',\n 'DHS20180000604',\n 'EG201405500107',\n 'EG201403610201',\n 'EG201404340105']"

In [20]:
gee_reduced_training.drop_duplicates(inplace = True, keep = 'first')
gee_reduced_training.shape

(10058, 2403)

In [21]:
# Find extra columns in gee_reduced_training
extra_columns_training = list(set(gee_reduced_training.columns) - set(gee_reduced_sample.columns))

# Find extra columns in gee_reduced_sample
extra_columns_sample = list(set(gee_reduced_sample.columns) - set(gee_reduced_training.columns))

# Print the extra column names
print("Extra columns in gee_reduced_training:", extra_columns_training)
print("Extra columns in gee_reduced_sample:", extra_columns_sample)

#Extra columns in gee_reduced_training: ['DHSCLUST', 'DHSYEAR']
#Extra columns in gee_reduced_sample: []

Extra columns in gee_reduced_training: ['DHSCLUST_x', 'DHSYEAR_x', 'DHSCLUST_y', 'DHSYEAR_y']
Extra columns in gee_reduced_sample: ['DHSCLUST', 'DHSYEAR']


In [26]:
gee_reduced_training.drop(extra_columns_training, inplace = True , axis = 1)
gee_reduced_sample.drop(extra_columns_sample, inplace = True , axis = 1)
print(gee_reduced_training.shape)
print(gee_reduced_sample.shape)
#(101138, 907)
#(14975, 907)

(10058, 2399)
(1587, 2399)


In [ ]:
#gee_reduced_training.to_csv('gee_reduced_training.csv' , index = False)
#gee_reduced_sample.to_csv('gee_reduced_sample.csv', index = False)

In [32]:
((gee_reduced_training.isnull())['Mean_BMI']).sum()

1957

In [35]:
gee_reduced_training.isnull().sum()

DHSID                                                                                              0
Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped       0
QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped                                                       0
SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped                                                   0
TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped                                                0
                                                                                                ... 
Median_BMI                                                                                      1957
Unmet_Need_Rate                                                                                  183
Under5_Mortality_Rate                                                                           2839
Skilled_Birth_Attendant_Rate                                                               

## Label-Wise Model Training

***Mean_BMI***

In [ ]:
#!gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
#!gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
#training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
#training

#here, training is same as gee_reduced_training

In [43]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates_1 = gee_reduced_training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates_1 = training_without_duplicates_1.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates_1


,DHSID,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,IA201400110884,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,IA201400051523,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,IA201400150534,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,NG200300000174,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,PH200800000205,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,IA201400320268,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,IA201400120410,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,EG200800000765,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,PE200000000614,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [44]:
dhsids_training_1 = training_without_duplicates_1['DHSID']
dhsids_training_1

0        IA201400110884
1        IA201400051523
2        IA201400150534
3        NG200300000174
4        PH200800000205
              ...      
10024    IA201400320268
10025    IA201400120410
10026    EG200800000765
10027    PE200000000614
10028    DR200700001622
Name: DHSID, Length: 10029, dtype: object

In [45]:
gee_reduced_training.columns[training_without_duplicates_1.isna().any() == True]

Index(['Mean_BMI', 'Median_BMI', 'Unmet_Need_Rate', 'Under5_Mortality_Rate',
       'Skilled_Birth_Attendant_Rate', 'Stunted_Rate'],
      dtype='object')

In [46]:
labels_1 = training_without_duplicates_1.iloc[:, -6:]
labels_1

,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,21.67,20.44,42.86,NaN,NaN,NaN
1,19.62,20.22,66.67,NaN,NaN,NaN
2,20.07,20.83,83.33,NaN,NaN,NaN
3,21.64,20.07,66.67,16.33,43.75,NaN
4,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...
10024,22.94,23.14,0.00,NaN,NaN,NaN
10025,22.67,21.05,5.56,NaN,NaN,NaN
10026,30.40,31.56,0.00,4.00,25.00,25.00
10027,25.70,24.13,17.65,10.53,4.55,22.22


In [47]:
for col in labels_1.columns:
    print(f'Null in {col}: {labels_1[col].isna().sum()}')
    
'''Null in Mean_BMI: 18491
Null in Median_BMI: 18491
Null in Unmet_Need_Rate: 1863
Null in Under5_Mortality_Rate: 28840
Null in Skilled_Birth_Attendant_Rate: 32476
Null in Stunted_Rate: 57209'''

Null in Mean_BMI: 1940
Null in Median_BMI: 1940
Null in Unmet_Need_Rate: 182
Null in Under5_Mortality_Rate: 2839
Null in Skilled_Birth_Attendant_Rate: 3188
Null in Stunted_Rate: 5735


'Null in Mean_BMI: 18491\nNull in Median_BMI: 18491\nNull in Unmet_Need_Rate: 1863\nNull in Under5_Mortality_Rate: 28840\nNull in Skilled_Birth_Attendant_Rate: 32476\nNull in Stunted_Rate: 57209'

**NOTE:** The original methodology we developed relied on the distinct characteristics of the training data, particularly randomly sampling rows. However, due to utilization of only 10% of the data in this context, the concept of row removal(sampling) loses its relevance

In [ ]:
import pandas as pd

# Assuming 'training_without_duplicates' is your DataFrame with features and labels

# Identify rows with null values in the 'Median_BMI' column
null_rows_1 = training_without_duplicates_1[training_without_duplicates_1['Mean_BMI'].isnull()]

# Set the random seed (random state) for reproducibility
random_state_1 = 42

# Number of rows to randomly remove
rows_to_remove_1 = 16000

# Get the indices of random null-valued rows to remove
rows_to_remove_indices_1 = null_rows_1.sample(n=rows_to_remove_1, random_state=random_state_1).index

# Remove the randomly selected null-valued rows
training_without_duplicates_1 = training_without_duplicates_1.drop(rows_to_remove_indices_1)

# Calculate the mean of 'Median_BMI' for imputation
avg_1 = training_without_duplicates_1['Mean_BMI'].mean()

# Impute remaining null values in 'Median_BMI' with the mean value
training_without_duplicates_1['Mean_BMI'].fillna(avg_1, inplace=True)

# The 'remaining_rows' DataFrame now contains the rows with non-null 'Median_BMI'
# after removing 16,000 random null-valued rows and imputing the remaining 2,000 null values with the mean.

In [ ]:
#sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
#sample

#here, sample is same as gee_reduced_sample

In [65]:
dhsids_sample_1 = gee_reduced_sample['DHSID']
dhsids_sample_1

0       ML200600000390
1       BO200800002157
2       TL201600000282
3       BF201000000006
4       NG200800000031
             ...      
1587    KE200800000234
1588    KE201400001195
1589    ML201200000249
1590    AM201500000295
1591    PE200000000348
Name: DHSID, Length: 1587, dtype: object

In [48]:
training_without_duplicates_1.drop('DHSID',axis = 1, inplace = True)
training_without_duplicates_1

,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,LAI_QA_flag_median@JAXA/GCOM-C/L3/LAND/LAI/V2&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,0.0,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,0.0,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,0.0,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,0.0,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,0.0,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,0.0,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,0.0,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,0.0,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,0.0,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


**NOTE:** As a means of substantiating the efficacy of our approach and demonstrating the functionality of the code, we are opting to comment out the hyperparameter tuning section. Instead, we are directly training a model on the data utilizing the optimal hyperparameter values we've obtained.

In [53]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
#import catboost as cb
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb


# Assuming 'training_imputed' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X_1 = training_without_duplicates_1.iloc[:, :-6]
y_1 = training_without_duplicates_1.loc[:, ['Mean_BMI']]

# # Define the number of folds for k-fold cross-validation
# n_folds_1 = 3

# # Define the k-fold cross-validator
# kf_1 = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1500),  # Expanded range
#         "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),  # Expanded range
#         "max_depth": trial.suggest_int("max_depth", 3, 20),  # Expanded range
#         "subsample": trial.suggest_float("subsample", 0.5, 1.0),  # Expanded range
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),  # Expanded range
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),  # Expanded range
#         "gamma": trial.suggest_float("gamma", 0.0, 5.0),  # Expanded range
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),  # Expanded range
#         "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),  # Expanded range
#         "random_state": 42,
#         "objective": "reg:squarederror",  # Regression task
#         "eval_metric": "rmse",  # Root Mean Squared Error metric
#         "tree_method": "gpu_hist"
#     }

#     # Initialize arrays to store the out-of-fold predictions and RMSE scores
#     oof_predictions = np.zeros(len(X))
#     rmses = []

#     for train_idx, val_idx in kf.split(X):
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         model = xgb.XGBRegressor(**params)
#         model.fit(X_train, y_train)
#         predictions = model.predict(X_val)

#         # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
#         oof_predictions[val_idx] = predictions

#         rmse = mean_squared_error(y_val, predictions, squared=False)
#         rmses.append(rmse)
        
#         # Report intermediate result for pruning
#         trial.report(rmse, step=len(rmses))
        
#         # Prune unpromising trials
#         if trial.should_prune():
#             raise optuna.TrialPruned()


#     # Calculate the overall RMSE for the k-fold cross-validation
#     cv_rmse = np.mean(rmses)

#     return cv_rmse


# study_xgb = optuna.create_study(direction='minimize')
# pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
# study_xgb.optimize(objective, n_trials=500)

# # After hyperparameter tuning, you can train the final model on the entire training data.
# # Use the best hyperparameters obtained from the Optuna study.

**NOTE:** The following code segment handles potential null values within the labels dataframe. However, it might not be necessary for execution when training on the complete dataset. Therefore, we kindly request to comment out this code while training the comprehensive model.

In [54]:
from sklearn.impute import SimpleImputer

# Create a SimpleImputer instance with mean strategy
imputer = SimpleImputer(strategy='mean')
columns = y_1.columns

# Replace missing values in DataFrame y_1 with mean values
y_1 = imputer.fit_transform(y_1)

# Convert the NumPy array back to a DataFrame (optional)
y_1 = pd.DataFrame(y_1, columns=columns)

# Now y_1_imputed_df contains the DataFrame with missing values filled using mean strategy

In [55]:
best_params_1 = {'n_estimators': 1021,
               'learning_rate': 0.007205299668408112,
               'max_depth': 10, 
               'subsample': 0.9883355319954155,
               'colsample_bytree': 0.3809596334310913,
               'min_child_weight': 12,
               'gamma': 0.0028943412540592683,
               'reg_alpha': 0.6123215976798917,
               'reg_lambda': 0.7185694163814978,
               "tree_method": "gpu_hist", 
               'random_state' : 42}

In [58]:
model_1 = xgb.XGBRegressor(**best_params_1)
model_1.fit(X_1, y_1)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.3809596334310913, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.0028943412540592683, gpu_id=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.007205299668408112, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=12, missing=nan, monotone_constraints=None,
             n_estimators=1021, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=42, ...)

In [59]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = gee_reduced_sample.drop('DHSID' , axis = 1)
X_sample_1 = sample2.iloc[:, :-6]
y_sample_1 = sample2.loc[:,'Mean_BMI']

In [60]:
# # Get the best hyperparameters for the current label
# best_params = study_xgb.best_params
# # Train a new model using the best hyperparameters for the current label
# final_xgboost_model = xgb.XGBRegressor(**best_params)
# final_xgboost_model.fit(X, y)
# # Make predictions for the current label
predictions_1 = model_1.predict(X_sample_1)

In [62]:
cols_1 = ["Mean_BMI"]

predictions_1 = pd.DataFrame(predictions_1 , columns = cols_1)
predictions_1

,Mean_BMI
0,22.437698
1,24.243561
2,20.705147
3,20.536955
4,21.594940
...,...
1582,24.059141
1583,23.180172
1584,22.059278
1585,24.213625


In [63]:
missing_dhsids_1 = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [66]:
df_1 = pd.DataFrame(dhsids_sample_1)
df_1

,DHSID
0,ML200600000390
1,BO200800002157
2,TL201600000282
3,BF201000000006
4,NG200800000031
...,...
1587,KE200800000234
1588,KE201400001195
1589,ML201200000249
1590,AM201500000295


In [67]:
pred_merge_df_1 = pd.concat([df_1, predictions_1], axis=1)

# Print the merged dataframe
pred_merge_df_1

,DHSID,Mean_BMI
0,ML200600000390,22.437698
1,BO200800002157,24.243561
2,TL201600000282,20.705147
3,BF201000000006,20.536955
4,NG200800000031,21.594940
...,...,...
10,NaN,27.559502
92,NaN,25.557487
166,NaN,30.185143
193,NaN,22.958387


In [68]:
missing_dhsid_df_1 = pd.DataFrame({'DHSID': missing_dhsids_1})

# Merge the DataFrames
merged_df_1 = pd.concat([missing_dhsid_df_1, pred_merge_df_1], axis=0)

# Print the merged DataFrame
merged_df_1

,DHSID,Mean_BMI
0,EG201403480201,NaN
1,EG201407490204,NaN
2,DHS20180000402,NaN
3,EG201406870404,NaN
4,EG201403980103,NaN
...,...,...
10,NaN,27.559502
92,NaN,25.557487
166,NaN,30.185143
193,NaN,22.958387


In [69]:
for column in merged_df_1.columns[1:] :
    merged_df_1[column] = merged_df_1[column].fillna(merged_df_1[column].mean())
merged_df_1

,DHSID,Mean_BMI
0,EG201403480201,23.794552
1,EG201407490204,23.794552
2,DHS20180000402,23.794552
3,EG201406870404,23.794552
4,EG201403980103,23.794552
...,...,...
10,NaN,27.559502
92,NaN,25.557487
166,NaN,30.185143
193,NaN,22.958387


In [70]:
merged_df_1.sort_values('DHSID' , inplace = True)
merged_df_1

,DHSID,Mean_BMI
805,AL200800000008,22.154383
573,AL200800000019,22.146170
1181,AL200800000026,23.219675
621,AL200800000085,21.489222
1050,AL200800000086,22.067478
...,...,...
10,NaN,27.559502
92,NaN,25.557487
166,NaN,30.185143
193,NaN,22.958387


In [72]:
merged_df_1.to_csv('XGB_(16k)kNull_RS42_MeanBMI.csv' , index = False)

In [73]:
# Reset index of merged_df_1
merged_df_1.reset_index(drop=True, inplace=True)

# Create final_df with columns 'DHSID' and 'Mean_BMI' from merged_df_1
final_df = merged_df_1[['DHSID', 'Mean_BMI']].copy()

# Now final_df contains the desired columns 'DHSID' and 'Mean_BMI'

***Median_BMI***

In [ ]:
# pip install gdown

In [ ]:
# !gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
# !gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
# training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
# training

In [75]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates_2 = gee_reduced_training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates_2 = training_without_duplicates_2.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates_2

,DHSID,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,IA201400110884,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,IA201400051523,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,IA201400150534,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,NG200300000174,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,PH200800000205,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,IA201400320268,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,IA201400120410,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,EG200800000765,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,PE200000000614,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [76]:
dhsids_training_2 = training_without_duplicates_2['DHSID']
dhsids_training_2

0        IA201400110884
1        IA201400051523
2        IA201400150534
3        NG200300000174
4        PH200800000205
              ...      
10024    IA201400320268
10025    IA201400120410
10026    EG200800000765
10027    PE200000000614
10028    DR200700001622
Name: DHSID, Length: 10029, dtype: object

In [ ]:
# training.columns[training_without_duplicates.isna().any() == True]

# '''Index(['Mean_BMI', 'Median_BMI', 'Unmet_Need_Rate', 'Under5_Mortality_Rate',
#        'Skilled_Birth_Attendant_Rate', 'Stunted_Rate'],
#       dtype='object')'''

In [77]:
labels_2 = training_without_duplicates_2.iloc[:, -6:]
labels_2

,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,21.67,20.44,42.86,NaN,NaN,NaN
1,19.62,20.22,66.67,NaN,NaN,NaN
2,20.07,20.83,83.33,NaN,NaN,NaN
3,21.64,20.07,66.67,16.33,43.75,NaN
4,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...
10024,22.94,23.14,0.00,NaN,NaN,NaN
10025,22.67,21.05,5.56,NaN,NaN,NaN
10026,30.40,31.56,0.00,4.00,25.00,25.00
10027,25.70,24.13,17.65,10.53,4.55,22.22


In [78]:
for col in labels_2.columns:
    print(f'Null in {col}: {labels_2[col].isna().sum()}')
    
'''Null in Mean_BMI: 18491
Null in Median_BMI: 18491
Null in Unmet_Need_Rate: 1863
Null in Under5_Mortality_Rate: 28840
Null in Skilled_Birth_Attendant_Rate: 32476
Null in Stunted_Rate: 57209'''

Null in Mean_BMI: 1940
Null in Median_BMI: 1940
Null in Unmet_Need_Rate: 182
Null in Under5_Mortality_Rate: 2839
Null in Skilled_Birth_Attendant_Rate: 3188
Null in Stunted_Rate: 5735


'Null in Mean_BMI: 18491\nNull in Median_BMI: 18491\nNull in Unmet_Need_Rate: 1863\nNull in Under5_Mortality_Rate: 28840\nNull in Skilled_Birth_Attendant_Rate: 32476\nNull in Stunted_Rate: 57209'

In [ ]:
import pandas as pd

# Assuming 'training_without_duplicates' is your DataFrame with features and labels

# Identify rows with null values in the 'Median_BMI' column
null_rows_2 = training_without_duplicates_2[training_without_duplicates_2['Median_BMI'].isnull()]

# Set the random seed (random state) for reproducibility
random_state_2 = 42

# Number of rows to randomly remove
rows_to_remove_2 = 16000

# Get the indices of random null-valued rows to remove
rows_to_remove_indices = null_rows_2.sample(n=rows_to_remove_2, random_state=random_state_2).index

# Remove the randomly selected null-valued rows
training_without_duplicates_2 = training_without_duplicates_2.drop(rows_to_remove_indices)

# Calculate the mean of 'Median_BMI' for imputation
avg_2 = training_without_duplicates_2['Median_BMI'].mean()

# Impute remaining null values in 'Median_BMI' with the mean value
training_without_duplicates_2['Median_BMI'].fillna(avg_2, inplace=True)

training_without_duplicates_2

# The 'remaining_rows' DataFrame now contains the rows with non-null 'Median_BMI'
# after removing 16,000 random null-valued rows and imputing the remaining 2,000 null values with the mean.

In [ ]:
# sample_2 = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
# sample_2

In [80]:
dhsids_sample_2 = gee_reduced_sample['DHSID']
dhsids_sample_2

0       ML200600000390
1       BO200800002157
2       TL201600000282
3       BF201000000006
4       NG200800000031
             ...      
1587    KE200800000234
1588    KE201400001195
1589    ML201200000249
1590    AM201500000295
1591    PE200000000348
Name: DHSID, Length: 1587, dtype: object

In [83]:
training_without_duplicates_2.drop('DHSID',axis = 1, inplace = True)
training_without_duplicates_2

,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,LAI_QA_flag_median@JAXA/GCOM-C/L3/LAND/LAI/V2&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,0.0,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,0.0,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,0.0,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,0.0,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,0.0,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,0.0,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,0.0,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,0.0,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,0.0,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [84]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
#import catboost as cb
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb


# Assuming 'training_imputed' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X_2 = training_without_duplicates_2.iloc[:, :-6]
y_2 = training_without_duplicates_2.loc[:, ['Median_BMI']]

# # Define the number of folds for k-fold cross-validation
# n_folds_2 = 3

# # Define the k-fold cross-validator
# kf_2 = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1500),  # Expanded range
#         "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),  # Expanded range
#         "max_depth": trial.suggest_int("max_depth", 3, 20),  # Expanded range
#         "subsample": trial.suggest_float("subsample", 0.5, 1.0),  # Expanded range
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),  # Expanded range
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),  # Expanded range
#         "gamma": trial.suggest_float("gamma", 0.0, 5.0),  # Expanded range
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),  # Expanded range
#         "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),  # Expanded range
#         "random_state": 42,
#         "objective": "reg:squarederror",  # Regression task
#         "eval_metric": "rmse",  # Root Mean Squared Error metric
#         "tree_method": "gpu_hist"
#     }

#     # Initialize arrays to store the out-of-fold predictions and RMSE scores
#     oof_predictions = np.zeros(len(X))
#     rmses = []

#     for train_idx, val_idx in kf.split(X):
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         model = xgb.XGBRegressor(**params)
#         model.fit(X_train, y_train)
#         predictions = model.predict(X_val)

#         # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
#         oof_predictions[val_idx] = predictions

#         rmse = mean_squared_error(y_val, predictions, squared=False)
#         rmses.append(rmse)
        
#         # Report intermediate result for pruning
#         trial.report(rmse, step=len(rmses))
        
#         # Prune unpromising trials
#         if trial.should_prune():
#             raise optuna.TrialPruned()


#     # Calculate the overall RMSE for the k-fold cross-validation
#     cv_rmse = np.mean(rmses)

#     return cv_rmse


# study_xgb = optuna.create_study(direction='minimize')
# pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
# study_xgb.optimize(objective, n_trials=500)

# # After hyperparameter tuning, you can train the final model on the entire training data.
# # Use the best hyperparameters obtained from the Optuna study.

In [85]:
from sklearn.impute import SimpleImputer

# Create a SimpleImputer instance with mean strategy
imputer = SimpleImputer(strategy='mean')
columns_2 = y_2.columns

# Replace missing values in DataFrame y_1 with mean values
y_2 = imputer.fit_transform(y_2)

# Convert the NumPy array back to a DataFrame (optional)
y_2 = pd.DataFrame(y_2, columns=columns_2)

# Now y_1_imputed_df contains the DataFrame with missing values filled using mean strategy

In [93]:
#Trial 408 finished with value: 2.1249550224769043
best_params_2 = {'n_estimators': 908, 
              'learning_rate': 0.008572596514064812, 
              'max_depth': 10,
              'subsample': 0.9806138553667393,
              'colsample_bytree': 0.3555037506225436,
              'min_child_weight': 17, 
              'gamma': 0.8512650980611904, 
              'reg_alpha': 0.5173019013846165, 
              'reg_lambda': 0.4812331516964748,
            "tree_method": "gpu_hist", 
             'random_state' : 42}

In [94]:
model_2 = xgb.XGBRegressor(**best_params_2)
model_2.fit(X_2, y_2)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.3555037506225436, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.8512650980611904, gpu_id=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.008572596514064812, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=17, missing=nan, monotone_constraints=None,
             n_estimators=908, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=42, ...)

In [95]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = gee_reduced_sample.drop('DHSID' , axis = 1)
X_sample_2 = sample2.iloc[:, :-6]
y_sample_2 = sample2.loc[:,'Median_BMI']

In [96]:
# # Get the best hyperparameters for the current label
# best_params = study_xgb.best_params
# # Train a new model using the best hyperparameters for the current label
# final_xgboost_model = xgb.XGBRegressor(**best_params)
# final_xgboost_model.fit(X, y)
# # Make predictions for the current label
predictions_2 = model_2.predict(X_sample_2)

In [97]:
cols_2 = ["Median_BMI"]

predictions_2 = pd.DataFrame(predictions_2 , columns = cols_2)
predictions_2

,Median_BMI
0,22.030117
1,24.139948
2,20.193619
3,20.361870
4,20.890404
...,...
1582,23.396011
1583,22.892284
1584,21.332075
1585,23.659407


In [98]:
missing_dhsids_2 = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [99]:
df_2 = pd.DataFrame(dhsids_sample_2)
df_2

,DHSID
0,ML200600000390
1,BO200800002157
2,TL201600000282
3,BF201000000006
4,NG200800000031
...,...
1587,KE200800000234
1588,KE201400001195
1589,ML201200000249
1590,AM201500000295


In [100]:
pred_merge_df_2 = pd.concat([df_2, predictions_2], axis=1)

# Print the merged dataframe
pred_merge_df_2

,DHSID,Median_BMI
0,ML200600000390,22.030117
1,BO200800002157,24.139948
2,TL201600000282,20.193619
3,BF201000000006,20.361870
4,NG200800000031,20.890404
...,...,...
10,NaN,25.488487
92,NaN,24.362175
166,NaN,30.228140
193,NaN,22.248255


In [102]:
missing_dhsid_df_2 = pd.DataFrame({'DHSID': missing_dhsids_2})

# Merge the DataFrames
merged_df_2 = pd.concat([missing_dhsid_df_2, pred_merge_df_2], axis=0)

# Print the merged DataFrame
merged_df_2

,DHSID,Median_BMI
0,EG201403480201,NaN
1,EG201407490204,NaN
2,DHS20180000402,NaN
3,EG201406870404,NaN
4,EG201403980103,NaN
...,...,...
10,NaN,25.488487
92,NaN,24.362175
166,NaN,30.228140
193,NaN,22.248255


In [103]:
for column in merged_df_2.columns[1:] :
    merged_df_2[column] = merged_df_2[column].fillna(merged_df_2[column].mean())
merged_df_2

,DHSID,Median_BMI
0,EG201403480201,23.328279
1,EG201407490204,23.328279
2,DHS20180000402,23.328279
3,EG201406870404,23.328279
4,EG201403980103,23.328279
...,...,...
10,NaN,25.488487
92,NaN,24.362175
166,NaN,30.228140
193,NaN,22.248255


In [104]:
merged_df_2.sort_values('DHSID' , inplace = True)
merged_df_2

,DHSID,Median_BMI
805,AL200800000008,21.462307
573,AL200800000019,21.644135
1181,AL200800000026,22.965000
621,AL200800000085,21.272259
1050,AL200800000086,21.776754
...,...,...
10,NaN,25.488487
92,NaN,24.362175
166,NaN,30.228140
193,NaN,22.248255


In [105]:
merged_df_2.to_csv('XGB_(18-2)kNull_RS42_MedianBMI.csv' , index = False)

In [107]:
merged_df_2.reset_index(drop=True, inplace=True)

In [108]:
final_df['Median_BMI'] = merged_df_2['Median_BMI']

In [109]:
final_df

,DHSID,Mean_BMI,Median_BMI
0,AL200800000008,22.154383,21.462307
1,AL200800000019,22.146170,21.644135
2,AL200800000026,23.219675,22.965000
3,AL200800000085,21.489222,21.272259
4,AL200800000086,22.067478,21.776754
...,...,...,...
1612,NaN,27.559502,25.488487
1613,NaN,25.557487,24.362175
1614,NaN,30.185143,30.228140
1615,NaN,22.958387,22.248255


***Unmet_Need_Rate***

In [ ]:
# !gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
# !gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
# training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
# training

In [110]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates_3 = gee_reduced_training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates_3 = training_without_duplicates_3.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates_3

,DHSID,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,IA201400110884,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,IA201400051523,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,IA201400150534,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,NG200300000174,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,PH200800000205,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,IA201400320268,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,IA201400120410,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,EG200800000765,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,PE200000000614,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [111]:
dhsids_training_3 = training_without_duplicates_3['DHSID']
dhsids_training_3

0        IA201400110884
1        IA201400051523
2        IA201400150534
3        NG200300000174
4        PH200800000205
              ...      
10024    IA201400320268
10025    IA201400120410
10026    EG200800000765
10027    PE200000000614
10028    DR200700001622
Name: DHSID, Length: 10029, dtype: object

In [ ]:
# training.columns[training_without_duplicates.isna().any() == True]

# '''Index(['Mean_BMI', 'Median_BMI', 'Unmet_Need_Rate', 'Under5_Mortality_Rate',
#        'Skilled_Birth_Attendant_Rate', 'Stunted_Rate'],
#       dtype='object')'''

In [112]:
labels_3 = training_without_duplicates_3.iloc[:, -6:]
labels_3

,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,21.67,20.44,42.86,NaN,NaN,NaN
1,19.62,20.22,66.67,NaN,NaN,NaN
2,20.07,20.83,83.33,NaN,NaN,NaN
3,21.64,20.07,66.67,16.33,43.75,NaN
4,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...
10024,22.94,23.14,0.00,NaN,NaN,NaN
10025,22.67,21.05,5.56,NaN,NaN,NaN
10026,30.40,31.56,0.00,4.00,25.00,25.00
10027,25.70,24.13,17.65,10.53,4.55,22.22


In [113]:
for col in labels_3.columns:
    print(f'{col}: {labels_3[col].isna().sum()}')
    
    
'''Mean_BMI: 18491
Median_BMI: 18491
Unmet_Need_Rate: 1863
Under5_Mortality_Rate: 28840
Skilled_Birth_Attendant_Rate: 32476
Stunted_Rate: 57209'''

Mean_BMI: 1940
Median_BMI: 1940
Unmet_Need_Rate: 182
Under5_Mortality_Rate: 2839
Skilled_Birth_Attendant_Rate: 3188
Stunted_Rate: 5735


'Mean_BMI: 18491\nMedian_BMI: 18491\nUnmet_Need_Rate: 1863\nUnder5_Mortality_Rate: 28840\nSkilled_Birth_Attendant_Rate: 32476\nStunted_Rate: 57209'

In [ ]:
# import numpy as np
# import pandas as pd

# # Assuming training_without_duplicates is your DataFrame
# # Remove the existing 1800 rows with null values
# training_without_duplicates_3.dropna(subset=['Unmet_Need_Rate'], inplace=True)

# # Remove an additional 2000 random rows using random state 42
# selected_rows = training_without_duplicates.sample(n=2000, random_state=42)
# training_without_duplicates_3.drop(selected_rows.index, inplace=True)

# # Impute any remaining null values with the mean
# avg_3 = training_without_duplicates_3['Unmet_Need_Rate'].mean()
# training_without_duplicates_3['Unmet_Need_Rate'].fillna(avg_3, inplace=True)

In [114]:
training_without_duplicates_3.shape
#(94348, 907)

(10029, 2399)

In [ ]:
# training.columns[training_without_duplicates.isna().any() == True]
# '''Index(['Mean_BMI', 'Median_BMI', 'Under5_Mortality_Rate',
#        'Skilled_Birth_Attendant_Rate', 'Stunted_Rate'],
#       dtype='object')'''

In [ ]:
# sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
# sample

In [116]:
dhsids_sample_3 = gee_reduced_sample['DHSID']
dhsids_sample_3

0       ML200600000390
1       BO200800002157
2       TL201600000282
3       BF201000000006
4       NG200800000031
             ...      
1587    KE200800000234
1588    KE201400001195
1589    ML201200000249
1590    AM201500000295
1591    PE200000000348
Name: DHSID, Length: 1587, dtype: object

In [117]:
training_without_duplicates_3.drop('DHSID',axis = 1, inplace = True)
training_without_duplicates_3

,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,LAI_QA_flag_median@JAXA/GCOM-C/L3/LAND/LAI/V2&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,0.0,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,0.0,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,0.0,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,0.0,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,0.0,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,0.0,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,0.0,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,0.0,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,0.0,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [120]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
#import catboost as cb
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb


# Assuming 'training_imputed' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X_3 = training_without_duplicates_3.iloc[:, :-6]
y_3 = training_without_duplicates_3.loc[:, ['Unmet_Need_Rate']]

# # Define the number of folds for k-fold cross-validation
# n_folds_3 = 3

# # Define the k-fold cross-validator
# kf_3 = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1500),  # Expanded range
#         "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),  # Expanded range
#         "max_depth": trial.suggest_int("max_depth", 3, 20),  # Expanded range
#         "subsample": trial.suggest_float("subsample", 0.5, 1.0),  # Expanded range
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),  # Expanded range
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),  # Expanded range
#         "gamma": trial.suggest_float("gamma", 0.0, 5.0),  # Expanded range
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),  # Expanded range
#         "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),  # Expanded range
#         "random_state": 42,
#         "objective": "reg:squarederror",  # Regression task
#         "eval_metric": "rmse",  # Root Mean Squared Error metric
#         "tree_method": "gpu_hist"
#     }

#     # Initialize arrays to store the out-of-fold predictions and RMSE scores
#     oof_predictions = np.zeros(len(X))
#     rmses = []

#     for train_idx, val_idx in kf.split(X):
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         model = xgb.XGBRegressor(**params)
#         model.fit(X_train, y_train)
#         predictions = model.predict(X_val)

#         # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
#         oof_predictions[val_idx] = predictions

#         rmse = mean_squared_error(y_val, predictions, squared=False)
#         rmses.append(rmse)
        
#         # Report intermediate result for pruning
#         trial.report(rmse, step=len(rmses))
        
#         # Prune unpromising trials
#         if trial.should_prune():
#             raise optuna.TrialPruned()


#     # Calculate the overall RMSE for the k-fold cross-validation
#     cv_rmse = np.mean(rmses)

#     return cv_rmse


# study_xgb = optuna.create_study(direction='minimize')
# pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
# study_xgb.optimize(objective, n_trials=500)

# # After hyperparameter tuning, you can train the final model on the entire training data.
# # Use the best hyperparameters obtained from the Optuna study

In [118]:
#Trial 123 finished with value: 19.07345502434539 
best_params_3 = {'n_estimators': 1494, 
               'learning_rate': 0.006122420799116568,
               'max_depth': 11, 
               'subsample': 0.9709483301720011, 
               'colsample_bytree': 0.5439318691481286, 
               'min_child_weight': 19,
               'gamma': 0.5836594670780896,
               'reg_alpha': 0.6926816720550927, 
               'reg_lambda': 0.057458000729569705,
               "tree_method": "gpu_hist",
               "random_state": 42
              }

In [122]:
from sklearn.impute import SimpleImputer

# Create a SimpleImputer instance with mean strategy
imputer = SimpleImputer(strategy='mean')
columns_3 = y_3.columns

# Replace missing values in DataFrame y_1 with mean values
y_3 = imputer.fit_transform(y_3)

# Convert the NumPy array back to a DataFrame (optional)
y_3 = pd.DataFrame(y_3, columns=columns_3)

In [123]:
model_3 = xgb.XGBRegressor(**best_params_3)
model_3.fit(X_3, y_3)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.5439318691481286, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.5836594670780896, gpu_id=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.006122420799116568, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=11, max_leaves=None,
             min_child_weight=19, missing=nan, monotone_constraints=None,
             n_estimators=1494, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=42, ...)

In [125]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = gee_reduced_sample.drop('DHSID' , axis = 1)
X_sample_3 = sample2.iloc[:, :-6]
y_sample_3 = sample2.loc[:,'Unmet_Need_Rate']

In [ ]:
# # Get the best hyperparameters for the current label
# best_params = study_xgb.best_params
# # Train a new model using the best hyperparameters for the current label
# final_xgboost_model = xgb.XGBRegressor(**best_params)
# final_xgboost_model.fit(X, y)

In [126]:
# Make predictions for the current label
predictions_3 = model_3.predict(X_sample_3)
predictions_3

array([59.283497, 30.665312, 56.849586, ..., 84.582664, 29.021317,
       20.20169 ], dtype=float32)

In [128]:
cols_3 = ["Unmet_Need_Rate"]

predictions_3 = pd.DataFrame(predictions_3 , columns = cols_3)
predictions_3

,Unmet_Need_Rate
0,59.283497
1,30.665312
2,56.849586
3,62.791496
4,86.993477
...,...
1582,26.433340
1583,33.328735
1584,84.582664
1585,29.021317


In [129]:
missing_dhsids_3 = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [130]:
df_3 = pd.DataFrame(dhsids_sample_3)
df_3

,DHSID
0,ML200600000390
1,BO200800002157
2,TL201600000282
3,BF201000000006
4,NG200800000031
...,...
1587,KE200800000234
1588,KE201400001195
1589,ML201200000249
1590,AM201500000295


In [131]:
pred_merge_df_3 = pd.concat([df_3, predictions_3], axis=1)

# Print the merged dataframe
pred_merge_df_3

,DHSID,Unmet_Need_Rate
0,ML200600000390,59.283497
1,BO200800002157,30.665312
2,TL201600000282,56.849586
3,BF201000000006,62.791496
4,NG200800000031,86.993477
...,...,...
10,NaN,23.927973
92,NaN,3.813371
166,NaN,38.215191
193,NaN,31.941771


In [132]:
missing_dhsid_df_3 = pd.DataFrame({'DHSID': missing_dhsids_3})

# Merge the DataFrames
merged_df_3 = pd.concat([missing_dhsid_df_3, pred_merge_df_3], axis=0)

# Print the merged DataFrame
merged_df_3

,DHSID,Unmet_Need_Rate
0,EG201403480201,NaN
1,EG201407490204,NaN
2,DHS20180000402,NaN
3,EG201406870404,NaN
4,EG201403980103,NaN
...,...,...
10,NaN,23.927973
92,NaN,3.813371
166,NaN,38.215191
193,NaN,31.941771


In [133]:
for column in merged_df_3.columns[1:] :
    merged_df_3[column] = merged_df_3[column].fillna(merged_df_3[column].mean())
merged_df_3

,DHSID,Unmet_Need_Rate
0,EG201403480201,35.648746
1,EG201407490204,35.648746
2,DHS20180000402,35.648746
3,EG201406870404,35.648746
4,EG201403980103,35.648746
...,...,...
10,NaN,23.927973
92,NaN,3.813371
166,NaN,38.215191
193,NaN,31.941771


In [134]:
merged_df_3.sort_values('DHSID' , inplace = True)
merged_df_3

,DHSID,Unmet_Need_Rate
805,AL200800000008,20.581995
573,AL200800000019,64.414764
1181,AL200800000026,43.585976
621,AL200800000085,56.057404
1050,AL200800000086,27.624147
...,...,...
10,NaN,23.927973
92,NaN,3.813371
166,NaN,38.215191
193,NaN,31.941771


In [135]:
merged_df_3.to_csv('XGB_Unmet_tuned_3kRowsRemoved_RS42.csv' , index = False)

In [136]:
merged_df_3.reset_index(drop=True, inplace=True)

In [137]:
final_df['Unmet_Need_Rate'] = merged_df_3['Unmet_Need_Rate']

In [138]:
final_df

,DHSID,Mean_BMI,Median_BMI,Unmet_Need_Rate
0,AL200800000008,22.154383,21.462307,20.581995
1,AL200800000019,22.146170,21.644135,64.414764
2,AL200800000026,23.219675,22.965000,43.585976
3,AL200800000085,21.489222,21.272259,56.057404
4,AL200800000086,22.067478,21.776754,27.624147
...,...,...,...,...
1612,NaN,27.559502,25.488487,23.927973
1613,NaN,25.557487,24.362175,3.813371
1614,NaN,30.185143,30.228140,38.215191
1615,NaN,22.958387,22.248255,31.941771


***Under5_Mortality_Rate***

In [ ]:
# training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)

In [139]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates_4 = gee_reduced_training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates_4 = training_without_duplicates_4.reset_index(drop=True)



In [140]:
training_without_duplicates_4.isnull().sum()

DHSID                                                                                              0
Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped       0
QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped                                                       0
SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped                                                   0
TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped                                                0
                                                                                                ... 
Median_BMI                                                                                      1940
Unmet_Need_Rate                                                                                  182
Under5_Mortality_Rate                                                                           2839
Skilled_Birth_Attendant_Rate                                                               

In [ ]:
# import pandas as pd
# from sklearn.impute import SimpleImputer
# import numpy as np

# # Assuming 'training_without_duplicates' is your DataFrame with features and labels

# # Randomly remove 22,000 rows with null values, using random state 42 for reproducibility
# null_rows = training_without_duplicates_4[training_without_duplicates_4['Under5_Mortality_Rate'].isnull()]
# random_null_rows = null_rows.sample(n=5000, random_state=42)
# training_without_duplicates_4 = training_without_duplicates_4.drop(random_null_rows.index)

# # Impute remaining null values using mean imputation
# imputer = SimpleImputer(strategy='mean')
# imputed_values = imputer.fit_transform(training_without_duplicates_4[['Under5_Mortality_Rate']])
# training_without_duplicates_4['Under5_Mortality_Rate'] = imputed_values

In [ ]:
# sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)

In [141]:
dhsids_sample_4 = gee_reduced_sample['DHSID']

In [142]:
training_without_duplicates_4.drop('DHSID',axis = 1, inplace = True)

In [159]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb

# Assuming 'training_without_duplicates' is your DataFrame after preprocessing

# Splitting the data into features (X) and labels (y)
X_4 = training_without_duplicates_4.iloc[:, :-6]  # Exclude the 'Skilled_Birth_Attendant_Rate' column
y_4 = training_without_duplicates_4[['Under5_Mortality_Rate']]

# # Define the number of folds for k-fold cross-validation
# n_folds_4 = 3

# # Define the k-fold cross-validator
# kf_4 = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
#         "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),
#         "max_depth": trial.suggest_int("max_depth", 3, 20),
#         "subsample": trial.suggest_float("subsample", 0.5, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
#         "gamma": trial.suggest_float("gamma", 0.0, 5.0),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
#         "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
#         "random_state": 42,
#         "objective": "reg:squarederror",
#         "eval_metric": "rmse",
#         "tree_method": "gpu_hist"
#     }

#     # Initialize arrays to store the out-of-fold predictions and RMSE scores
#     oof_predictions = np.zeros(len(X))
#     rmses = []

#     for train_idx, val_idx in kf.split(X):
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         model = xgb.XGBRegressor(**params)
#         model.fit(X_train, y_train)
#         predictions = model.predict(X_val)

#         # Store the out-of-fold predictions for potential ensemble techniques
#         oof_predictions[val_idx] = predictions

#         rmse = mean_squared_error(y_val, predictions, squared=False)
#         rmses.append(rmse)
        
#         # Report intermediate result for pruning
#         trial.report(rmse, step=len(rmses))
        
#         # Prune unpromising trials
#         if trial.should_prune():
#             raise optuna.TrialPruned()

#     # Calculate the overall RMSE for the k-fold cross-validation
#     cv_rmse = np.mean(rmses)

#     return cv_rmse

# study_xgb = optuna.create_study(direction='minimize')
# study_xgb.optimize(objective, n_trials=400)

# # After hyperparameter tuning, you can train the final model on the entire training data.
# # Use the best hyperparameters obtained from the Optuna study.


In [152]:
#Trial 82 finished with value: 4.766250927332784 
best_params_4 = {'n_estimators': 1279,
               'learning_rate': 0.008244534414963565,
               'max_depth': 10, 
               'subsample': 0.9977006392316401, 
               'colsample_bytree': 0.34089086689526177, 
               'min_child_weight': 19, 
               'gamma': 0.39657616554247066,
               'reg_alpha': 0.5993129929719778, 
               'reg_lambda': 0.6119614886920933,
                "tree_method": "gpu_hist",
               'random_state' : 42}

In [160]:
from sklearn.impute import SimpleImputer

# Create a SimpleImputer instance with mean strategy
imputer = SimpleImputer(strategy='mean')
columns_4 = y_4.columns

# Replace missing values in DataFrame y_1 with mean values
y_4 = imputer.fit_transform(y_4)

# Convert the NumPy array back to a DataFrame (optional)
y_4 = pd.DataFrame(y_4, columns=columns_4)

In [163]:
model_4 = xgb.XGBRegressor(**best_params_4)
model_4.fit(X_4, y_4)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.34089086689526177, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.39657616554247066, gpu_id=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.008244534414963565, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=19, missing=nan, monotone_constraints=None,
             n_estimators=1279, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=42, ...)

In [161]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = gee_reduced_sample.drop('DHSID' , axis = 1)
X_sample_4 = sample2.iloc[:, :-6]
y_sample_4 = sample2.loc[:,'Under5_Mortality_Rate']

In [164]:
# # Get the best hyperparameters for the current label
# best_params = study_xgb.best_params
# # Train a new model using the best hyperparameters for the current label
# final_xgboost_model = xgb.XGBRegressor(**best_params)
# final_xgboost_model.fit(X, y)
# # Make predictions for the current label
predictions_4 = model_4.predict(X_sample_4)

In [165]:
cols_4 = ["Under5_Mortality_Rate"]

predictions_4 = pd.DataFrame(predictions_4 , columns = cols_4)

In [166]:
missing_dhsids_4 = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [167]:
df_4 = pd.DataFrame(dhsids_sample_4)

In [170]:
pred_merge_df_4 = pd.concat([df_4, predictions_4], axis=1)

In [171]:
missing_dhsid_df_4 = pd.DataFrame({'DHSID': missing_dhsids_4})

# Merge the DataFrames
merged_df_4 = pd.concat([missing_dhsid_df_4, pred_merge_df_4], axis=0)

In [172]:
for column in merged_df_4.columns[1:] :
    merged_df_4[column] = merged_df_4[column].fillna(merged_df_4[column].mean())

In [173]:
merged_df_4.sort_values('DHSID' , inplace = True)

In [174]:
merged_df_4.to_csv('XGB_5K_null_Under5_Mortality_Rate_tuned_org.csv' , index = False)

In [175]:
merged_df_4.reset_index(drop=True, inplace=True)

In [176]:
final_df['Under5_Mortality_Rate'] = merged_df_4['Under5_Mortality_Rate']

In [177]:
final_df

,DHSID,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate
0,AL200800000008,22.154383,21.462307,20.581995,10.916060
1,AL200800000019,22.146170,21.644135,64.414764,15.508948
2,AL200800000026,23.219675,22.965000,43.585976,10.055983
3,AL200800000085,21.489222,21.272259,56.057404,11.118331
4,AL200800000086,22.067478,21.776754,27.624147,7.091116
...,...,...,...,...,...
1612,NaN,27.559502,25.488487,23.927973,3.662781
1613,NaN,25.557487,24.362175,3.813371,1.642945
1614,NaN,30.185143,30.228140,38.215191,7.830527
1615,NaN,22.958387,22.248255,31.941771,8.485593


***Skilled_Birth_Attendant_Rate***

In [ ]:
# !gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
# !gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
# train_data = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
# train_data

In [178]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates_5 = gee_reduced_training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates_5 = training_without_duplicates_5.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates_5


,DHSID,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,IA201400110884,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,IA201400051523,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,IA201400150534,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,NG200300000174,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,PH200800000205,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,IA201400320268,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,IA201400120410,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,EG200800000765,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,PE200000000614,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [179]:
dhsids_training_5 = training_without_duplicates_5['DHSID']
dhsids_training_5

0        IA201400110884
1        IA201400051523
2        IA201400150534
3        NG200300000174
4        PH200800000205
              ...      
10024    IA201400320268
10025    IA201400120410
10026    EG200800000765
10027    PE200000000614
10028    DR200700001622
Name: DHSID, Length: 10029, dtype: object

In [ ]:
# training.columns[training_without_duplicates.isna().any() == True]

In [180]:
labels_5 = training_without_duplicates_5.iloc[:, -6:]
labels_5

,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,21.67,20.44,42.86,NaN,NaN,NaN
1,19.62,20.22,66.67,NaN,NaN,NaN
2,20.07,20.83,83.33,NaN,NaN,NaN
3,21.64,20.07,66.67,16.33,43.75,NaN
4,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...
10024,22.94,23.14,0.00,NaN,NaN,NaN
10025,22.67,21.05,5.56,NaN,NaN,NaN
10026,30.40,31.56,0.00,4.00,25.00,25.00
10027,25.70,24.13,17.65,10.53,4.55,22.22


In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.impute import SimpleImputer

# # Step 3: Perform KNN imputation to fill NaN values
# imputer = SimpleImputer(strategy='mean')  # You can adjust the n_neighbors parameter as needed
# training_without_duplicates_5[labels.columns] = imputer.fit_transform(labels_5)

# # Step 4: Verify the imputed values in 'training' DataFrame
# training_without_duplicates_5

In [ ]:
# sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
# sample

In [181]:
dhsids_sample_5 = gee_reduced_sample['DHSID']
dhsids_sample_5

0       ML200600000390
1       BO200800002157
2       TL201600000282
3       BF201000000006
4       NG200800000031
             ...      
1587    KE200800000234
1588    KE201400001195
1589    ML201200000249
1590    AM201500000295
1591    PE200000000348
Name: DHSID, Length: 1587, dtype: object

In [182]:
training_without_duplicates_5.drop('DHSID',axis = 1, inplace = True)
training_without_duplicates_5

,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,LAI_QA_flag_median@JAXA/GCOM-C/L3/LAND/LAI/V2&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,0.0,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,0.0,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,0.0,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,0.0,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,0.0,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,0.0,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,0.0,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,0.0,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,0.0,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [183]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import optuna

# Assuming 'training_imputed' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X_5 = training_without_duplicates_5.iloc[:, :-6]
y_5 = training_without_duplicates_5.loc[:, ['Skilled_Birth_Attendant_Rate']]

# # Define the number of folds for k-fold cross-validation
# n_folds_5 = 5

# # Define the k-fold cross-validator
# kf_5 = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1200),  # Expanded range
#         "learning_rate": trial.suggest_float("learning_rate", 1e-5, 0.2, log=True),
#         "max_depth": trial.suggest_int("max_depth", 3, 20),
#         "subsample": trial.suggest_float("subsample", 0.1, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
#         "gamma": trial.suggest_float("gamma", 0.0, 5.0),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
#         "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
#         "random_state": 42,
#         "objective": "reg:squarederror",  # Regression task
#         "eval_metric": "rmse", # Root Mean Squared Error metric
#         "tree_method": "gpu_hist"
#         # Add more XGBoost parameters here for tuning (if needed)
#         # For example: 'alpha', 'lambda', 'min_split_loss', etc.
#     }

#     # Initialize arrays to store the out-of-fold predictions and RMSE scores
#     oof_predictions = np.zeros(len(X))
#     rmses = []

#     for train_idx, val_idx in kf.split(X):
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         model = xgb.XGBRegressor(**params)
#         model.fit(X_train, y_train)
#         predictions = model.predict(X_val)

#         # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
#         oof_predictions[val_idx] = predictions

#         rmse = mean_squared_error(y_val, predictions, squared=False)
#         rmses.append(rmse)
        
#         # Report intermediate result for pruning
#         trial.report(rmse, step=len(rmses))
        
#         # Prune unpromising trials
#         if trial.should_prune():
#             raise optuna.TrialPruned()


#     # Calculate the overall RMSE for the k-fold cross-validation
#     cv_rmse = np.mean(rmses)

#     return cv_rmse


# study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler())
# pruner = optuna.pruners.MedianPruner(n_warmup_steps=5)
# study_xgb.optimize(objective, n_trials=400)

# # After hyperparameter tuning, you can train the final model on the entire training data.
# # Use the best hyperparameters obtained from the Optuna study.


In [184]:
best_params_5 =  {'n_estimators': 1148, 
                'learning_rate': 0.009970261274385907,
                 'max_depth': 13, 
                 'subsample': 0.9636818032665447, 
                 'colsample_bytree': 0.6296043325752896, 
                 'min_child_weight': 18,
                 'gamma': 1.970249558007862, 
                 'reg_alpha': 0.5772114039696489,
                 'reg_lambda': 0.18376992370296957,
                 'tree_method' : 'gpu_hist',
                'random_state' : 42}

In [ ]:
# print('Best hyperparameters:', study_xgb.best_params)
# print('Best RMSE:', study_xgb.best_value)

In [185]:
from sklearn.impute import SimpleImputer

# Create a SimpleImputer instance with mean strategy
imputer = SimpleImputer(strategy='mean')
columns_5 = y_5.columns

# Replace missing values in DataFrame y_1 with mean values
y_5 = imputer.fit_transform(y_5)

# Convert the NumPy array back to a DataFrame (optional)
y_5 = pd.DataFrame(y_5, columns=columns_5)

In [186]:
model_5 = xgb.XGBRegressor(**best_params_5)
model_5.fit(X_5, y_5)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6296043325752896, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=1.970249558007862, gpu_id=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.009970261274385907, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=13, max_leaves=None,
             min_child_weight=18, missing=nan, monotone_constraints=None,
             n_estimators=1148, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=42, ...)

In [187]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = gee_reduced_sample.drop('DHSID' , axis = 1)
X_sample_5 = sample2.iloc[:, :-6]
y_sample_5 = sample2.loc[:,'Skilled_Birth_Attendant_Rate']

In [188]:
# # Get the best hyperparameters for the current label
# best_params = study_xgb.best_params
# # Train a new model using the best hyperparameters for the current label
# final_xgboost_model = xgb.XGBRegressor(**best_params)
# final_xgboost_model.fit(X, y)
# # Make predictions for the current label
predictions_5 = model_5.predict(X_sample_5)

In [189]:
cols_5 = ["Skilled_Birth_Attendant_Rate"]

predictions_5 = pd.DataFrame(predictions_5 , columns = cols_5)
predictions_5

,Skilled_Birth_Attendant_Rate
0,71.678520
1,51.314228
2,36.600277
3,52.687534
4,10.747849
...,...
1582,71.394791
1583,54.945942
1584,8.259273
1585,93.718697


In [190]:
missing_dhsids_5 = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [191]:
df_5 = pd.DataFrame(dhsids_sample_5)
df_5

,DHSID
0,ML200600000390
1,BO200800002157
2,TL201600000282
3,BF201000000006
4,NG200800000031
...,...
1587,KE200800000234
1588,KE201400001195
1589,ML201200000249
1590,AM201500000295


In [192]:
pred_merge_df_5 = pd.concat([df_5, predictions_5], axis=1)

# Print the merged dataframe
pred_merge_df_5

,DHSID,Skilled_Birth_Attendant_Rate
0,ML200600000390,71.678520
1,BO200800002157,51.314228
2,TL201600000282,36.600277
3,BF201000000006,52.687534
4,NG200800000031,10.747849
...,...,...
10,NaN,97.087662
92,NaN,89.485497
166,NaN,83.107956
193,NaN,53.668240


In [193]:
missing_dhsid_df_5 = pd.DataFrame({'DHSID': missing_dhsids_5})

# Merge the DataFrames
merged_df_5 = pd.concat([missing_dhsid_df_5, pred_merge_df_5], axis=0)

# Print the merged DataFrame
merged_df_5

,DHSID,Skilled_Birth_Attendant_Rate
0,EG201403480201,NaN
1,EG201407490204,NaN
2,DHS20180000402,NaN
3,EG201406870404,NaN
4,EG201403980103,NaN
...,...,...
10,NaN,97.087662
92,NaN,89.485497
166,NaN,83.107956
193,NaN,53.668240


In [194]:
for column in merged_df_5.columns[1:] :
    merged_df_5[column] = merged_df_5[column].fillna(merged_df_5[column].mean())
merged_df_5

,DHSID,Skilled_Birth_Attendant_Rate
0,EG201403480201,67.799232
1,EG201407490204,67.799232
2,DHS20180000402,67.799232
3,EG201406870404,67.799232
4,EG201403980103,67.799232
...,...,...
10,NaN,97.087662
92,NaN,89.485497
166,NaN,83.107956
193,NaN,53.668240


In [195]:
merged_df_5.sort_values('DHSID' , inplace = True)
merged_df_5

,DHSID,Skilled_Birth_Attendant_Rate
805,AL200800000008,49.918064
573,AL200800000019,55.381950
1181,AL200800000026,67.336533
621,AL200800000085,15.883365
1050,AL200800000086,78.180389
...,...,...
10,NaN,97.087662
92,NaN,89.485497
166,NaN,83.107956
193,NaN,53.668240


In [196]:
merged_df_5.to_csv('iterSearchsp_900Cols-XGboost-OptunaHyper-Skilled_Birth_Attendant_Rate-CrossVal-GPU.csv' , index = False)

In [197]:
merged_df_5.reset_index(drop=True, inplace=True)

In [198]:
final_df['Skilled_Birth_Attendant_Rate'] = merged_df_5['Skilled_Birth_Attendant_Rate']

***Stunted_Rate***

In [ ]:
# !gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
# !gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
# training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
# training

In [199]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates_6 = gee_reduced_training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates_6 = training_without_duplicates_6.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates_6

,DHSID,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,IA201400110884,50.0,16513.3630,-2.997677,934.0,64.916664,-1.0,138.0,-2.99828,-1.000000,...,1533.6666,1554.0000,-2.965393,0.303478,21.67,20.44,42.86,NaN,NaN,NaN
1,IA201400051523,208.0,16514.0780,-2.997677,811.5,64.416664,-1.0,140.0,-2.99828,-1.000000,...,1130.8334,1329.0000,-2.965393,-0.266114,19.62,20.22,66.67,NaN,NaN,NaN
2,IA201400150534,156.5,16514.0940,-1.437762,821.0,79.583336,-1.0,136.0,-2.99828,14.082428,...,1233.5000,888.0000,-0.835440,-0.171740,20.07,20.83,83.33,NaN,NaN,NaN
3,NG200300000174,245.0,16514.0530,-2.997677,1369.0,112.250000,-1.0,139.0,-2.99828,-1.000000,...,1589.7277,2677.0000,-2.965393,-0.255735,21.64,20.07,66.67,16.33,43.75,NaN
4,PH200800000205,173.0,16514.4280,-2.997677,600.0,34.500000,-1.0,140.0,-2.99828,-1.000000,...,1243.2500,1857.0000,-2.965393,-0.360048,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10024,IA201400320268,180.0,16514.3960,-2.997677,786.0,72.666664,-1.0,140.0,-2.99828,-1.000000,...,1031.5000,793.0000,-2.965393,0.210895,22.94,23.14,0.00,NaN,NaN,NaN
10025,IA201400120410,142.0,128.9823,-2.997677,482.0,99.083336,-1.0,139.0,-2.99828,-1.000000,...,1372.4166,41.0000,-2.965393,0.028165,22.67,21.05,5.56,NaN,NaN,NaN
10026,EG200800000765,197.5,16513.9180,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.000000,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,PE200000000614,161.5,16514.4060,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.000000,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [200]:
dhsids_training_6 = training_without_duplicates_6['DHSID']
dhsids_training_6

0        IA201400110884
1        IA201400051523
2        IA201400150534
3        NG200300000174
4        PH200800000205
              ...      
10024    IA201400320268
10025    IA201400120410
10026    EG200800000765
10027    PE200000000614
10028    DR200700001622
Name: DHSID, Length: 10029, dtype: object

In [ ]:
# training.columns[training_without_duplicates.isna().any() == True]

In [201]:
labels_6 = training_without_duplicates_6.iloc[:, -6:]
labels_6

,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,21.67,20.44,42.86,NaN,NaN,NaN
1,19.62,20.22,66.67,NaN,NaN,NaN
2,20.07,20.83,83.33,NaN,NaN,NaN
3,21.64,20.07,66.67,16.33,43.75,NaN
4,NaN,NaN,33.33,2.78,87.50,NaN
...,...,...,...,...,...,...
10024,22.94,23.14,0.00,NaN,NaN,NaN
10025,22.67,21.05,5.56,NaN,NaN,NaN
10026,30.40,31.56,0.00,4.00,25.00,25.00
10027,25.70,24.13,17.65,10.53,4.55,22.22


In [202]:
for col in labels_6.columns:
    print(f'{col}: {labels_6[col].isna().sum()}')

Mean_BMI: 1940
Median_BMI: 1940
Unmet_Need_Rate: 182
Under5_Mortality_Rate: 2839
Skilled_Birth_Attendant_Rate: 3188
Stunted_Rate: 5735


In [203]:
training_without_duplicates_6.dropna(subset=['Stunted_Rate'], inplace = True)

In [204]:
training_without_duplicates_6.shape

(4294, 2399)

In [ ]:
# sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
# sample

In [205]:
dhsids_sample_6 = gee_reduced_sample['DHSID']
dhsids_sample_6

0       ML200600000390
1       BO200800002157
2       TL201600000282
3       BF201000000006
4       NG200800000031
             ...      
1587    KE200800000234
1588    KE201400001195
1589    ML201200000249
1590    AM201500000295
1591    PE200000000348
Name: DHSID, Length: 1587, dtype: object

In [206]:
training_without_duplicates_6.drop('DHSID',axis = 1, inplace = True)
training_without_duplicates_6

,Cloud_Effective_Radius_Undetermined_Std_Deviation_Mean_median@MODIS/061/MOD08_M3&timestamped,QA_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,SummaryQA_kurtosis@MODIS/006/MYD13A1&timestamped,TIMEOFDAY_median@NOAA/CDR/AVHRR/NDVI/V5&timestamped,Cloud_Effective_Radius_37_PCL_Liquid_Mean_Uncertainty_mean@MODIS/061/MOD08_M3&timestamped,line_number_stdDev@FIRMS&timestamped,Day_view_time_max_max@MODIS/006/MYD11A2&timestamped,sur_refl_b03_kurtosis@MODIS/006/MOD13A2&timestamped,NDVI_stdDev@NOAA/CDR/AVHRR/NDVI/V5&timestamped,LAI_QA_flag_median@JAXA/GCOM-C/L3/LAND/LAI/V2&timestamped,...,pet_mean@IDAHO_EPSCOR/TERRACLIMATE&timestamped,ViewZenith_median@MODIS/006/MOD13A2&timestamped,SO2_column_number_density_15km_kurtosis@COPERNICUS/S5P/NRTI/L3_SO2&timestamped,ssma_mean@NASA_USDA/HSL/soil_moisture&timestamped,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
7,-1.0,16514.352,-2.997677,819.0,16.333334,-1.0,131.0,-2.99828,-1.0,0.0,...,941.6782,1371.7467,-2.965393,0.309695,23.25,22.26,28.57,6.82,40.00,66.67
8,190.5,16514.340,-2.997677,1211.0,35.833332,-1.0,131.0,-2.99828,-1.0,0.0,...,1087.3239,2185.0000,-2.965393,0.545860,22.06,19.27,20.00,18.06,66.67,33.33
11,289.5,16514.470,-2.997677,791.0,65.000000,-1.0,138.0,-2.99828,-1.0,0.0,...,1062.4194,2620.0000,-2.965393,-0.221994,22.68,22.12,16.67,10.17,40.00,10.00
17,197.5,16514.043,-2.997677,1501.5,85.250000,-1.0,140.0,-2.99828,-1.0,0.0,...,1297.4529,2057.0000,-2.965393,0.123932,20.71,20.18,27.27,18.60,95.45,41.67
18,207.0,16514.000,-2.997677,1162.0,18.500000,-1.0,141.0,-2.99828,-1.0,0.0,...,1020.0276,2520.0537,-2.965393,0.794862,20.26,20.03,25.00,13.95,22.73,58.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10021,139.0,16515.920,-2.997677,1424.0,59.833332,-1.0,138.0,-2.99828,-1.0,0.0,...,1208.5000,2644.0000,-2.965393,-3.095048,28.60,24.67,28.57,0.00,100.00,20.00
10023,187.0,16515.300,-2.997677,1170.5,54.833332,-1.0,143.0,-2.99828,-1.0,0.0,...,1117.7035,2405.0000,-2.965393,-3.095048,25.13,24.98,100.00,8.62,0.00,22.22
10026,197.5,16513.918,-2.997677,1194.0,66.833336,-1.0,135.0,-2.99828,-1.0,0.0,...,1529.9166,471.0000,-2.965393,0.006586,30.40,31.56,0.00,4.00,25.00,25.00
10027,161.5,16514.406,-2.997677,1953.5,21.000000,-1.0,142.0,-2.99828,-1.0,0.0,...,900.8954,3529.0000,-2.965393,0.076976,25.70,24.13,17.65,10.53,4.55,22.22


In [209]:
import pandas as pd
import cupy as cp
import numpy as np
from cuml.ensemble import RandomForestRegressor as cuRandomForestRegressor
from cuml.metrics import mean_squared_error as mean_squared_error_cuml
from sklearn.model_selection import KFold
import optuna

# Assuming 'training_without_duplicates' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X_6 = training_without_duplicates_6.iloc[:, :-6]
y_6 = training_without_duplicates_6.loc[:, 'Stunted_Rate']

# Convert data to cuDF and cuML-compatible format
X_cudf_6 = cp.asarray(X_6.values.astype(np.float32))
y_cudf_6 = cp.asarray(y_6.values.astype(np.float32))

# # Define the number of folds for k-fold cross-validation
# n_folds_6 = 3

# # Define the k-fold cross-validator
# kf_6 = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
#         "max_depth": trial.suggest_int("max_depth", 5, 20),
#         "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
#         "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
#         "max_features": trial.suggest_float("max_features", 0.1, 1.0),
#         "random_state": 42,
#         "n_streams": 1
#     }

#     # Initialize arrays to store the out-of-fold predictions and RMSE scores
#     oof_predictions = cp.zeros(len(X_cudf))
#     rmses = cp.zeros(n_folds)

#     for i, (train_idx, val_idx) in enumerate(kf.split(X_cudf)):
#         X_train, X_val = X_cudf[train_idx], X_cudf[val_idx]
#         y_train, y_val = y_cudf[train_idx], y_cudf[val_idx]

#         model = cuRandomForestRegressor(**params)
#         model.fit(X_train, y_train)

#         predictions = model.predict(X_val)

#         # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
#         oof_predictions[val_idx] = predictions

#         rmse = cp.sqrt(mean_squared_error_cuml(y_val, predictions))
#         rmses[i] = rmse

#         # Prune the trial if the RMSE is too high (early stopping)
#         trial.report(float(rmse), step=i)
#         if trial.should_prune():
#             raise optuna.exceptions.TrialPruned()

#     # Calculate the overall RMSE for the k-fold cross-validation
#     cv_rmse = cp.mean(rmses)

#     return cv_rmse

# study_rf = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
# study_rf.optimize(objective, n_trials=250)

# # Access the best hyperparameters found by Optuna
# best_params = study_rf.best_params

# print("Best Hyperparameters:")
# print(best_params)

In [210]:
best_params_6 = {'n_estimators': 997, 
               'max_depth': 20,
               'min_samples_split': 6,
               'min_samples_leaf': 10, 
               'max_features': 0.6031911007542703,
               'random_state': 42
              }

In [ ]:
# print('Best hyperparameters:', study_rf.best_params)
# print('Best RMSE:', study_rf.best_value)

In [ ]:
# best_params = study_rf.best_params

In [211]:
# Train the final Random Forest model on the entire training data using the best hyperparameters
model_6 = cuRandomForestRegressor(**best_params_6)
model_6.fit(X_cudf_6, y_cudf_6)

/opt/conda/lib/python3.10/site-packages/cuml/internals/api_decorators.py:344: UserWarning: For reproducible results in Random Forest Classifier or for almost reproducible results in Random Forest Regressor, n_streams=1 is recommended. If n_streams is > 1, results may vary due to stream/thread timing differences, even when random_state is set
  return func(**kwargs)


RandomForestRegressor()

In [213]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = gee_reduced_sample.drop('DHSID' , axis = 1)
X_sample_6 = sample2.iloc[:, :-6]
y_sample_6 = sample2.loc[:,'Stunted_Rate']

In [215]:
# Make predictions for the current label
predictions_6 = model_6.predict(X_sample_6)
predictions_6

0       27.358438
1       36.647373
2       50.575920
3       34.162899
4       51.020397
          ...    
1587    23.129211
1588    35.052113
1589    42.481079
1590    26.288357
1591    39.304825
Length: 1587, dtype: float32

In [216]:
cols_6 = ["Stunted_Rate"]

predictions_6 = pd.DataFrame(predictions_6 , columns = cols_6)
predictions_6

,Stunted_Rate
0,27.358438
1,36.647373
2,50.575920
3,34.162899
4,51.020397
...,...
1587,23.129211
1588,35.052113
1589,42.481079
1590,26.288357


In [217]:
missing_dhsids_6 = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [218]:
df_6 = pd.DataFrame(dhsids_sample_6)
df_6

,DHSID
0,ML200600000390
1,BO200800002157
2,TL201600000282
3,BF201000000006
4,NG200800000031
...,...
1587,KE200800000234
1588,KE201400001195
1589,ML201200000249
1590,AM201500000295


In [219]:
pred_merge_df_6 = pd.concat([df_6, predictions_6], axis=1)

# Print the merged dataframe
pred_merge_df_6

,DHSID,Stunted_Rate
0,ML200600000390,27.358438
1,BO200800002157,36.647373
2,TL201600000282,50.575920
3,BF201000000006,34.162899
4,NG200800000031,51.020397
...,...,...
1587,KE200800000234,23.129211
1588,KE201400001195,35.052113
1589,ML201200000249,42.481079
1590,AM201500000295,26.288357


In [220]:
missing_dhsid_df_6 = pd.DataFrame({'DHSID': missing_dhsids_6})

# Merge the DataFrames
merged_df_6 = pd.concat([missing_dhsid_df_6, pred_merge_df_6], axis=0)

# Print the merged DataFrame
merged_df_6

,DHSID,Stunted_Rate
0,EG201403480201,NaN
1,EG201407490204,NaN
2,DHS20180000402,NaN
3,EG201406870404,NaN
4,EG201403980103,NaN
...,...,...
1587,KE200800000234,23.129211
1588,KE201400001195,35.052113
1589,ML201200000249,42.481079
1590,AM201500000295,26.288357


In [221]:
for column in merged_df_6.columns[1:] :
    merged_df_6[column] = merged_df_6[column].fillna(merged_df_6[column].mean())
merged_df_6

,DHSID,Stunted_Rate
0,EG201403480201,29.663517
1,EG201407490204,29.663517
2,DHS20180000402,29.663517
3,EG201406870404,29.663517
4,EG201403980103,29.663517
...,...,...
1587,KE200800000234,23.129211
1588,KE201400001195,35.052113
1589,ML201200000249,42.481079
1590,AM201500000295,26.288357


In [222]:
merged_df_6.sort_values('DHSID' , inplace = True)
merged_df_6

,DHSID,Stunted_Rate
805,AL200800000008,18.073887
573,AL200800000019,23.712536
1181,AL200800000026,24.622143
621,AL200800000085,32.619141
1050,AL200800000086,21.335049
...,...,...
1111,ZW201500000282,31.494112
435,ZW201500000289,30.758411
504,ZW201500000325,21.268253
875,ZW201500000371,27.218542


In [223]:
merged_df_6.to_csv('RF_Stunted_Rate_tuned_NoNullRowsRemoved.csv' , index = False)

In [225]:
merged_df_6.reset_index(drop = True, inplace = True)

In [226]:
final_df['Stunted_Rate'] = merged_df_6['Stunted_Rate']

In [227]:
final_df

,DHSID,Mean_BMI,Median_BMI,Unmet_Need_Rate,Under5_Mortality_Rate,Skilled_Birth_Attendant_Rate,Stunted_Rate
0,AL200800000008,22.154383,21.462307,20.581995,10.916060,49.918064,18.073887
1,AL200800000019,22.146170,21.644135,64.414764,15.508948,55.381950,23.712536
2,AL200800000026,23.219675,22.965000,43.585976,10.055983,67.336533,24.622143
3,AL200800000085,21.489222,21.272259,56.057404,11.118331,15.883365,32.619141
4,AL200800000086,22.067478,21.776754,27.624147,7.091116,78.180389,21.335049
...,...,...,...,...,...,...,...
1612,NaN,27.559502,25.488487,23.927973,3.662781,97.087662,NaN
1613,NaN,25.557487,24.362175,3.813371,1.642945,89.485497,NaN
1614,NaN,30.185143,30.228140,38.215191,7.830527,83.107956,NaN
1615,NaN,22.958387,22.248255,31.941771,8.485593,53.668240,NaN


In [228]:
final_df.to_csv('final_labelwise_predictions.csv')